# conv-channel-sum — worked example 2: Convolve two IC halves separately and sum the results

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-channel-sum`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Because conv2d sums over the in-channel axis, the contribution of one block of input channels is *additive* with the contribution of any other block. Splitting the IC axis into two groups, convolving each group with its matching kernel slice, and adding the two outputs reproduces the full convolution exactly. This is just associativity of the IC sum.

## Worked solution

We exploit the linearity of the IC contraction: `sum_{ic} a_ic = sum_{ic in A} a_ic + sum_{ic in B} a_ic`.

**Step 1 — pick a split point.** With `IC` input channels, choose `s = IC // 2`. Channels `[0:s]` form group A, channels `[s:IC]` form group B.

**Step 2 — slice both x and weight on the IC axis together.** The IC axis is axis 1 of `x` and axis 1 of `weight`. They must be sliced identically: `x[:, :s]` pairs with `weight[:, :s]`, and `x[:, s:]` pairs with `weight[:, s:]`. Mismatching the slices would contract the wrong channels.

**Step 3 — convolve each group.** `F.conv2d(x[:, :s], weight[:, :s])` sums only over group-A channels; the second call sums only over group-B channels. Each call already produces a full `(B, OC, OH, OW)` output because OC and the spatial axes are untouched by the split.

**Step 4 — add the two partial sums.** Adding `yA + yB` completes the sum over all IC. Because every output position is `sum over all ic`, partitioning that sum and re-adding gives the identical result — confirming the channel sum is the only thing the IC axis does.

In [ ]:
import torch.nn.functional as F

def conv2d_split_ic(x, weight):
    IC = x.shape[1]
    s = IC // 2
    yA = F.conv2d(x[:, :s], weight[:, :s])
    yB = F.conv2d(x[:, s:], weight[:, s:])
    return yA + yB

t.manual_seed(0)
x = t.randn(2, 4, 8, 8)
weight = t.randn(5, 4, 3, 3)
y = conv2d_split_ic(x, weight)
ref = F.conv2d(x, weight)
print(y.shape, t.allclose(y, ref, atol=1e-5))